In [1]:
import numpy as np
import tensorflow as tf

In [2]:
data = np.loadtxt('./data.csv', delimiter=',', unpack=True, dtype='float32')
print(data)

[[ 0.  1.  1.  0.  0.  0.]
 [ 0.  0.  1.  0.  0.  1.]
 [ 1.  0.  0.  1.  1.  0.]
 [ 0.  1.  0.  0.  0.  0.]
 [ 0.  0.  1.  0.  0.  1.]]


In [3]:
x_data = np.transpose(data[0:2])
print(x_data)

y_data = np.transpose(data[2:])
print(y_data)

[[ 0.  0.]
 [ 1.  0.]
 [ 1.  1.]
 [ 0.  0.]
 [ 0.  0.]
 [ 0.  1.]]
[[ 1.  0.  0.]
 [ 0.  1.  0.]
 [ 0.  0.  1.]
 [ 1.  0.  0.]
 [ 1.  0.  0.]
 [ 0.  0.  1.]]


In [4]:
global_step = tf.Variable(0, trainable=False, name='global_step')

In [5]:
X = tf.placeholder(tf.float32, [None, 2])
Y = tf.placeholder(tf.float32, [None, 3])

W1 = tf.Variable(tf.random_uniform([2, 10], -1., 1.))
W2 = tf.Variable(tf.random_uniform([10, 20], -1., 1.))
W3 = tf.Variable(tf.random_uniform([20, 3], -1., 1.))

In [6]:
L1 = tf.nn.relu(tf.matmul(X, W1))
L2 = tf.nn.relu(tf.matmul(L1, W2))
model = tf.matmul(L2, W3)

In [7]:
cost = tf.reduce_mean(tf.nn.softmax_cross_entropy_with_logits(labels=Y, logits=model))

train_op = tf.train.AdamOptimizer(learning_rate=0.01).minimize(cost, global_step=global_step)

In [8]:
sess = tf.Session()
saver = tf.train.Saver(tf.global_variables())

In [9]:
ckpt = tf.train.get_checkpoint_state('./model')

if ckpt and tf.train.checkpoint_exists(ckpt.model_checkpoint_path):
    saver.restore(sess, ckpt.model_checkpoint_path)
else:
    sess.run(tf.global_variables_initializer())

INFO:tensorflow:Restoring parameters from ./model\dnn.ckpt-2


In [10]:
for step in range(2):
    sess.run(train_op, feed_dict={X: x_data, Y: y_data})
    
    print('Step: %d' % sess.run(global_step), 'Cost: %.3f' % sess.run(cost, feed_dict={X: x_data, Y: y_data}))

Step: 3 Cost: 1.055
Step: 4 Cost: 0.980


In [11]:
saver.save(sess, './model/dnn.ckpt', global_step=global_step)

'./model/dnn.ckpt-4'

In [12]:
prediction = tf.argmax(model, 1)
target = tf.argmax(Y, 1)

print('Prediction:', sess.run(prediction, feed_dict={X: x_data}))
print('Target:', sess.run(target, feed_dict={Y: y_data}))

Prediction: [0 0 2 0 0 2]
Target: [0 1 2 0 0 2]


In [13]:
is_correct = tf.equal(prediction, target)
accuracy = tf.reduce_mean(tf.cast(is_correct, tf.float32))

print('Accuracy: %2.f' % sess.run(accuracy * 100, feed_dict={X: x_data, Y: y_data}))

Accuracy: 83
